In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import re
from tqdm.auto import tqdm

def locate_root():
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists():
            return candidate
    return current

root = locate_root()
metadata_dir = root / "data" / "metadata"
inventory = pd.read_csv(metadata_dir / "file_inventory.csv")
sheet_inventory = pd.read_csv(metadata_dir / "sheet_inventory.csv")
inventory.head()

/home/hamza/Documents/aero-res/aircraft_fault_localization/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,file_name,relative_path,folder,suffix,size_bytes,group
0,Defect15(01).xlsx,data/raw/defective/Defect15(01).xlsx,defective,.xlsx,63799916,defective
1,Defect15(02).xlsx,data/raw/defective/Defect15(02).xlsx,defective,.xlsx,6907967,defective
2,Defect15(03).xlsx,data/raw/defective/Defect15(03).xlsx,defective,.xlsx,77841167,defective
3,Defect15(04).xlsx,data/raw/defective/Defect15(04).xlsx,defective,.xlsx,40512363,defective
4,Defect15.xlsx,data/raw/defective/Defect15.xlsx,defective,.xlsx,7637,defective


In [2]:
def normalize_columns(columns):
    normalized = []
    for value in columns:
        text = "" if pd.isna(value) else str(value).strip().lower()
        text = re.sub(r"\s+", "_", text)
        text = re.sub(r"[^a-z0-9_]", "", text)
        normalized.append(text if text else "unnamed")
    seen = {}
    output = []
    for name in normalized:
        seen[name] = seen.get(name, 0) + 1
        output.append(name if seen[name] == 1 else f"{name}_{seen[name]}")
    return output

def candidate_sheet(file_name):
    sub = sheet_inventory[sheet_inventory["file_name"] == file_name].copy()
    sub["max_row"] = pd.to_numeric(sub["max_row"], errors="coerce")
    sub["max_column"] = pd.to_numeric(sub["max_column"], errors="coerce")
    sub = sub.sort_values(["max_row", "max_column"], ascending=[False, False])
    if sub.empty:
        return None
    return sub.iloc[0]["sheet_name"]

candidate_sheet_map = {name: candidate_sheet(name) for name in inventory["file_name"]}
inventory["candidate_sheet"] = inventory["file_name"].map(candidate_sheet_map)
inventory[["file_name", "group", "candidate_sheet"]]

,file_name,group,candidate_sheet
0,Defect15(01).xlsx,defective,08180820
1,Defect15(02).xlsx,defective,01151347
2,Defect15(03).xlsx,defective,01140926
3,Defect15(04).xlsx,defective,02091140
4,Defect15.xlsx,defective,Sheet1
5,Defect16(01).xlsx,defective,05141609
6,Defect16(02).xlsx,defective,02181728
7,Defect16(03).xlsx,defective,05060630
8,Defected 1.xlsx,defective,Defect_Flight_D4
9,Defected 2.xlsx,defective,Defect_Flight_D7


In [3]:
def choose_header_from_preview(preview):
    preview = preview.dropna(axis=1, how="all").dropna(axis=0, how="all")
    if preview.empty:
        return None
    best_header = None
    best_score = -1
    ncols = preview.shape[1]
    for h in [0, 1, 2, 3]:
        if h >= len(preview):
            continue
        cols = normalize_columns(preview.iloc[h].tolist())
        score = sum(not str(col).startswith("unnamed") for col in cols)
        if score > best_score:
            best_score = score
            best_header = h
    if best_header is None:
        return None
    if best_score < max(1, int(ncols * 0.4)):
        return None
    return best_header

def read_flight_frame(path, sheet_name, preview_rows=6):
    with pd.ExcelFile(path, engine="openpyxl") as xls:
        preview = pd.read_excel(xls, sheet_name=sheet_name, header=None, nrows=preview_rows)
        header = choose_header_from_preview(preview)
        df = pd.read_excel(xls, sheet_name=sheet_name, header=header)
    df = df.dropna(axis=1, how="all").dropna(axis=0, how="all")
    df.columns = normalize_columns(df.columns)
    return df, header

work_inventory = inventory[inventory["group"].isin(["healthy", "defective"])].copy()
work_inventory = work_inventory[work_inventory["candidate_sheet"].notna()].reset_index(drop=True)

profiles = []

for row in tqdm(work_inventory.itertuples(index=False), total=len(work_inventory)):
    path = root / row.relative_path
    try:
        df, header = read_flight_frame(path, row.candidate_sheet)
        sample = df if len(df) <= 2000 else df.sample(2000, random_state=42)
        profiles.append(
            {
                "file_name": row.file_name,
                "group": row.group,
                "relative_path": row.relative_path,
                "sheet_name": row.candidate_sheet,
                "header_row": header,
                "rows": int(df.shape[0]),
                "columns": int(df.shape[1]),
                "numeric_columns": int(df.select_dtypes(include=np.number).shape[1]),
                "missing_ratio": float(df.isna().mean().mean()),
                "duplicate_ratio_sample": float(sample.duplicated().mean()),
                "column_list": "|".join(map(str, df.columns)),
                "error": None,
            }
        )
    except Exception as e:
        profiles.append(
            {
                "file_name": row.file_name,
                "group": row.group,
                "relative_path": row.relative_path,
                "sheet_name": row.candidate_sheet,
                "header_row": None,
                "rows": None,
                "columns": None,
                "numeric_columns": None,
                "missing_ratio": None,
                "duplicate_ratio_sample": None,
                "column_list": None,
                "error": str(e),
            }
        )

profile_df = pd.DataFrame(profiles)
profile_df

100%|██████████| 24/24 [11:01<00:00, 27.58s/it]


,file_name,group,relative_path,sheet_name,header_row,rows,columns,numeric_columns,missing_ratio,duplicate_ratio_sample,column_list,error
0,Defect15(01).xlsx,defective,data/raw/defective/Defect15(01).xlsx,08180820,0.0,12431,208,189,0.0,0.0,pws|egpws|ns2|xdl|ny|sigmaxr|sigmay|sigmazr|si...,None
1,Defect15(02).xlsx,defective,data/raw/defective/Defect15(02).xlsx,01151347,0.0,4902,208,189,0.0,0.0,pws|egpws|ns2|xdl|ny|sigmaxr|sigmay|sigmazr|si...,None
2,Defect15(03).xlsx,defective,data/raw/defective/Defect15(03).xlsx,01140926,0.0,15689,208,189,0.0,0.0,pws|egpws|ns2|xdl|ny|sigmaxr|sigmay|sigmazr|si...,None
3,Defect15(04).xlsx,defective,data/raw/defective/Defect15(04).xlsx,02091140,0.0,13123,208,189,0.0,0.0,pws|egpws|ns2|xdl|ny|sigmaxr|sigmay|sigmazr|si...,None
4,Defect15.xlsx,defective,data/raw/defective/Defect15.xlsx,Sheet1,NaN,0,0,0,NaN,NaN,,None
5,Defect16(01).xlsx,defective,data/raw/defective/Defect16(01).xlsx,05141609,0.0,9690,208,189,0.0,0.0,pws|egpws|ns2|xdl|ny|sigmaxr|sigmay|sigmazr|si...,None
6,Defect16(02).xlsx,defective,data/raw/defective/Defect16(02).xlsx,02181728,0.0,14537,208,189,0.0,0.0,pws|egpws|ns2|xdl|ny|sigmaxr|sigmay|sigmazr|si...,None
7,Defect16(03).xlsx,defective,data/raw/defective/Defect16(03).xlsx,05060630,0.0,14683,208,189,0.0,0.0,pws|egpws|ns2|xdl|ny|sigmaxr|sigmay|sigmazr|si...,None
8,Defected 1.xlsx,defective,data/raw/defective/Defected 1.xlsx,Defect_Flight_D4,0.0,12431,208,189,0.0,0.0,pws|egpws|ns2|xdl|ny|sigmaxr|sigmay|sigmazr|si...,None
9,Defected 2.xlsx,defective,data/raw/defective/Defected 2.xlsx,Defect_Flight_D7,0.0,4902,208,189,0.0,0.0,pws|egpws|ns2|xdl|ny|sigmaxr|sigmay|sigmazr|si...,None


In [4]:
successful_profiles = profile_df[profile_df["error"].isna()].copy()

def split_columns(value):
    if pd.isna(value) or str(value).strip() == "":
        return []
    return [col for col in str(value).split("|") if col]

column_sets = {
    row.file_name: set(split_columns(row.column_list))
    for row in successful_profiles.itertuples(index=False)
}

healthy_files = successful_profiles.loc[successful_profiles["group"] == "healthy", "file_name"].tolist()
defective_files = successful_profiles.loc[successful_profiles["group"] == "defective", "file_name"].tolist()

healthy_common = sorted(set.intersection(*(column_sets[name] for name in healthy_files))) if healthy_files else []
defective_common = sorted(set.intersection(*(column_sets[name] for name in defective_files))) if defective_files else []
all_common = sorted(set.intersection(*(column_sets[name] for name in column_sets))) if column_sets else []

pd.DataFrame(
    {
        "metric": [
            "healthy_files",
            "defective_files",
            "healthy_common_columns",
            "defective_common_columns",
            "all_common_columns",
            "errored_files",
        ],
        "value": [
            len(healthy_files),
            len(defective_files),
            len(healthy_common),
            len(defective_common),
            len(all_common),
            int(profile_df["error"].notna().sum()),
        ],
    }
)

,metric,value
0,healthy_files,9
1,defective_files,15
2,healthy_common_columns,196
3,defective_common_columns,0
4,all_common_columns,0
5,errored_files,0


In [5]:
column_frequency = (
    pd.DataFrame(
        [
            {"file_name": row.file_name, "group": row.group, "column_name": col}
            for row in successful_profiles.itertuples(index=False)
            for col in split_columns(row.column_list)
        ]
    )
    .groupby("column_name", as_index=False)
    .agg(
        file_count=("file_name", "nunique"),
        healthy_file_count=("group", lambda s: int((s == "healthy").sum())),
        defective_file_count=("group", lambda s: int((s == "defective").sum())),
    )
    .sort_values(["file_count", "column_name"], ascending=[False, True])
    .reset_index(drop=True)
)

column_frequency

,column_name,file_count,healthy_file_count,defective_file_count
0,2ffkg123,23,9,14
1,2ffkg4,23,9,14
2,2ffy1,23,9,14
3,2ffy2,23,9,14
4,2ffy3,23,9,14
...,...,...,...,...
216,efis_sw,1,1,0
217,fxdl,1,1,0
218,gxqygd,1,1,0
219,sdl,1,1,0


In [6]:
quality_summary = (
    successful_profiles.groupby("group", dropna=False)
    .agg(
        files=("file_name", "nunique"),
        median_rows=("rows", "median"),
        median_columns=("columns", "median"),
        median_numeric_columns=("numeric_columns", "median"),
        avg_missing_ratio=("missing_ratio", "mean"),
        avg_duplicate_ratio_sample=("duplicate_ratio_sample", "mean"),
    )
    .reset_index()
)

quality_summary

,group,files,median_rows,median_columns,median_numeric_columns,avg_missing_ratio,avg_duplicate_ratio_sample
0,defective,15,13123.0,208.0,189.0,0.0,0.0
1,healthy,9,18263.0,208.0,189.0,0.0,0.0


In [7]:
profile_df.to_csv(metadata_dir / "dataset_profile.csv", index=False)
column_frequency.to_csv(metadata_dir / "column_frequency.csv", index=False)
pd.DataFrame({"column_name": all_common}).to_csv(metadata_dir / "common_columns_all_files.csv", index=False)
pd.DataFrame({"column_name": healthy_common}).to_csv(metadata_dir / "common_columns_healthy.csv", index=False)
pd.DataFrame({"column_name": defective_common}).to_csv(metadata_dir / "common_columns_defective.csv", index=False)
quality_summary.to_csv(metadata_dir / "quality_summary.csv", index=False)

{
    "dataset_profile": str(metadata_dir / "dataset_profile.csv"),
    "column_frequency": str(metadata_dir / "column_frequency.csv"),
    "common_columns_all_files": str(metadata_dir / "common_columns_all_files.csv"),
    "common_columns_healthy": str(metadata_dir / "common_columns_healthy.csv"),
    "common_columns_defective": str(metadata_dir / "common_columns_defective.csv"),
    "quality_summary": str(metadata_dir / "quality_summary.csv"),
}

{'dataset_profile': '/home/hamza/Documents/aero-res/aircraft_fault_localization/data/metadata/dataset_profile.csv',
 'column_frequency': '/home/hamza/Documents/aero-res/aircraft_fault_localization/data/metadata/column_frequency.csv',
 'common_columns_all_files': '/home/hamza/Documents/aero-res/aircraft_fault_localization/data/metadata/common_columns_all_files.csv',
 'common_columns_healthy': '/home/hamza/Documents/aero-res/aircraft_fault_localization/data/metadata/common_columns_healthy.csv',
 'common_columns_defective': '/home/hamza/Documents/aero-res/aircraft_fault_localization/data/metadata/common_columns_defective.csv',
 'quality_summary': '/home/hamza/Documents/aero-res/aircraft_fault_localization/data/metadata/quality_summary.csv'}